# Лабораторная работа №20: Мультимодальные агенты (Vision + Language + Action)

**ФИО студента:** _______________________
**Группа:** _______________________

## Цель работы
Изучить архитектуру мультимодальных агентов, объединяющих зрение (VLM), язык (LLM) и действия (Tools). Научиться создавать агентов, способных анализировать изображения, рассуждать и выполнять последовательные действия на основе визуального контекста.

## Теоретические сведения
### Мультимодальные агенты
Мультимодальный агент — это система, которая может воспринимать информацию через разные модальности (текст, изображения, аудио) и выполнять действия для достижения целей.

### Компоненты архитектуры
1. **Vision Encoder:** Извлечение признаков из изображений (CLIP, ViT)
2. **Language Model:** Рассуждение и планирование (LLM)
3. **Tool Interface:** Выполнение действий (API, функции)
4. **Memory:** Хранение контекста и истории

### Паттерны работы
- **See-Think-Act:** Восприятие → Рассуждение → Действие
- **Chain of Thought:** Пошаговое объяснение решений
- **ReAct:** Чередование рассуждений и действий

## Задание
### Уровень 1 (Базовый)
1. Подключите мультимодальную модель (Qwen-VL или LLaVA)
2. Реализуйте простое описание изображения
3. Создайте инструмент для OCR (распознавание текста)

### Уровень 2 (Продвинутый)
1. Реализуйте многошаговое рассуждение (Chain of Thought)
2. Создайте агента для решения визуальных головоломок
3. Добавьте инструменты для детекции объектов

### Уровень 3 (Индивидуальный)
1. Создайте агента-ассистента для анализа скриншотов интерфейсов
2. Реализуйте планирование последовательности действий
3. Добавьте память для контекста между запросами

## Ход работы

### Шаг 1: Установка зависимостей

In [ ]:
%pip install langchain langchain-openai pillow requests torch torchvision -q
%pip install transformers accelerate -q

### Шаг 2: Загрузка тестового изображения

In [ ]:
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# Загружаем тестовое изображение
image_url = "https://picsum.photos/seed/test123/800/600.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

plt.figure(figsize=(12, 9))
plt.imshow(image)
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Размер изображения: {image.size}")

### Шаг 3: Подключение мультимодальной модели

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage
import base64

# os.environ["OPENAI_API_KEY"] = "your-key-here"

# GPT-4 Vision поддерживает анализ изображений
llm_vision = ChatOpenAI(model="gpt-4-vision-preview", max_tokens=1024)

# Кодирование изображения в base64
def image_to_base64(img):
    buffered = BytesIO()
    img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode()

image_b64 = image_to_base64(image)

### Шаг 4: Базовое описание изображения

In [ ]:
# Создание сообщения с изображением
message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "Подробно опиши, что изображено на этой картинке. Какие объекты ты видишь?"
        },
        {
            "type": "image_url",
            "image_url": f"data:image/jpeg;base64,{image_b64}"
        }
    ]
)

# В реальном режиме:
# response = llm_vision.invoke([message])
# print(response.content)

# Демонстрация
print("=== ОПИСАНИЕ ИЗОБРАЖЕНИЯ ===")
print("На изображении представлено...")
print("Основные объекты: здания, деревья, люди...")
print("Цветовая гамма: теплые тона, дневное освещение...")

### Шаг 5: Создание инструментов для агента

In [ ]:
from langchain.tools import tool

@tool
def describe_scene(image_data: str) -> str:
    """Описывает сцену на изображении"""
    # В реальности здесь вызов VLM
    return "Сцена содержит городскую застройку с парком"

@tool
def detect_objects(image_data: str) -> list:
    """Детектирует объекты на изображении"""
    # В реальности здесь вызов модели детекции
    return [
        {"object": "здание", "confidence": 0.95},
        {"object": "дерево", "confidence": 0.89},
        {"object": "человек", "confidence": 0.76}
    ]

@tool
def read_text(image_data: str) -> str:
    """Распознаёт текст на изображении (OCR)"""
    # В реальности здесь вызов OCR модели
    return "Улица Ленина, дом 42\nМагазин 'Продукты'"

@tool
def find_object_position(image_data: str, object_name: str) -> dict:
    """Находит позицию объекта на изображении"""
    # В реальности здесь вызов модели с bounding boxes
    return {
        "found": True,
        "bbox": [100, 150, 300, 400],
        "position": "центр изображения"
    }

tools = [describe_scene, detect_objects, read_text, find_object_position]
print(f"Создано {len(tools)} инструментов для агента")

### Шаг 6: Агент с Chain of Thought

In [ ]:
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# Промпт для агента с инструкцией о пошаговом рассуждении
prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты мультимодальный агент с возможностями анализа изображений.
    
    ВАЖНО: Всегда рассуждай пошагово перед действием!
    1. Сначала проанализируй запрос
    2. Определи, какие инструменты нужны
    3. Выполни инструменты по очереди
    4. Синтезируй окончательный ответ
    
    Используй формат Chain of Thought для объяснения своих действий.
    """),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Создание агента (упрощённая версия для демо)
class SimpleMultimodalAgent:
    def __init__(self, tools):
        self.tools = {t.name: t for t in tools}
    
    def think_and_act(self, query, image_data):
        print(f"\n=== ЗАПРОС: {query}\n")
        
        # Рассуждение
        print("🤔 РАССУЖДЕНИЕ:")
        if "объект" in query.lower() and "где" in query.lower():
            print("  Пользователь спрашивает о позиции объекта.")
            print("  Нужно использовать инструмент find_object_position.")
            
            # Извлечение имени объекта
            obj_name = query.split("'")[1] if "'" in query else "объект"
            print(f"  Целевой объект: {obj_name}")
            
            # Действие
            print("\n⚡ ДЕЙСТВИЕ:")
            result = self.tools['find_object_position'].invoke({
                "image_data": image_data,
                "object_name": obj_name
            })
            print(f"  Результат: {result}")
            
            # Ответ
            print("\n✅ ОТВЕТ:")
            return f"Объект '{obj_name}' найден в позиции: {result['position']}"
        
        elif "текст" in query.lower() or "надпись" in query.lower():
            print("  Пользователь хочет прочитать текст на изображении.")
            print("  Использую инструмент read_text (OCR).")
            
            print("\n⚡ ДЕЙСТВИЕ:")
            result = self.tools['read_text'].invoke({"image_data": image_data})
            print(f"  Распознанный текст: {result}")
            
            print("\n✅ ОТВЕТ:")
            return f"На изображении обнаружен текст: {result}"
        
        else:
            print("  Общее описание сцены.")
            print("  Использую инструмент describe_scene.")
            
            print("\n⚡ ДЕЙСТВИЕ:")
            result = self.tools['describe_scene'].invoke({"image_data": image_data})
            print(f"  Описание: {result}")
            
            print("\n✅ ОТВЕТ:")
            return result

agent = SimpleMultimodalAgent(tools)

### Шаг 7: Тестирование агента

In [ ]:
# Тестовые запросы
queries = [
    "Опиши, что ты видишь на этом изображении?",
    "Где находится объект 'здание' на картинке?",
    "Какой текст можно прочитать на этом изображении?",
    "Найди все объекты и их позиции"
]

for query in queries:
    response = agent.think_and_act(query, image_b64)
    print(f"\n{'='*50}\n")

### Шаг 8: Многошаговый агент для визуальных головоломок

In [ ]:
def solve_visual_puzzle(image_data, puzzle_description):
    """Агент для решения многошаговых визуальных задач"""
    print(f"\n🧩 ГОЛОВОЛОМКА: {puzzle_description}\n")
    
    steps = []
    
    # Шаг 1: Общее описание
    print("Шаг 1: Анализ общей сцены...")
    scene = tools[0].invoke({"image_data": image_data})
    steps.append(f"Сцена: {scene}")
    
    # Шаг 2: Детекция объектов
    print("Шаг 2: Детекция объектов...")
    objects = tools[1].invoke({"image_data": image_data})
    steps.append(f"Объекты: {[o['object'] for o in objects]}")
    
    # Шаг 3: Поиск конкретных объектов
    print("Шаг 3: Поиск целевых объектов...")
    target_obj = "здание"
    position = tools[3].invoke({"image_data": image_data, "object_name": target_obj})
    steps.append(f"Позиция '{target_obj}': {position['position']}")
    
    # Шаг 4: Синтез ответа
    print("\n🎯 РЕШЕНИЕ:")
    solution = f"""
    На основе анализа изображения:
    1. {steps[0]}
    2. Обнаружены объекты: {', '.join([o['object'] for o in objects])}
    3. {steps[2]}
    
    Ответ: {target_obj} находится {position['position']}.
    """
    return solution

# Пример сложного запроса
puzzle = "Найди самое высокое здание и определи, что находится слева от него"
solution = solve_visual_puzzle(image_b64, puzzle)
print(solution)

## Контрольные вопросы
1. Какие преимущества у мультимодальных агентов перед текстовыми?
2. Как реализовать память контекста между запросами?
3. Какие проблемы возникают при комбинировании VLM и LLM?
4. Как оптимизировать количество вызовов моделей для снижения стоимости?

## Выводы
В работе изучена архитектура мультимодальных агентов. Реализованы инструменты для анализа изображений, детекции объектов и OCR. Показан паттерн See-Think-Act с Chain of Thought для многошагового рассуждения. Создан прототип агента для решения визуальных задач.